# 🕵️‍♂️ (Solution) Adventure in Amianta

## The Amianta Financial Services Scandal
You are an exiled data analyst tasked with uncovering a deep-rooted financial conspiracy within the city-state of Amianta. For months, rumors have swirled about a clandestine figure known only as **The Master**, who is said to be manipulating the state's financial systems to fund illicit projects and channel wealth to offshore accounts.

Your task is to uncover the scale of this corruption.

Don't forget to put on your "Anonymous". Without it, you're not a real hacker!

## Setup and Configuration
First, let's import the necessary libraries and set up our connection. **Replace the placeholder values below** with the `PORT` and `PASSWORD` from your Amianta server's console output.

Make sure that the server is running in the background. You can accomplish this by running the ```subprocess``` command.

```python
import subprocess

subprocess.Popen(["python", "gov_server.py"])
```

In [1]:
!pip3 install setproctitle requests


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import re
import requests
from dataclasses import dataclass
from typing import Tuple
from functools import reduce, partial
from itertools import product, chain
from concurrent.futures import ThreadPoolExecutor
import string

In [3]:
PORT = 38663  # 👈 Replace with the actual port
PASSWORD = "s2p" #" # 👈 Replace with the actual password

---

## Phase 0: Server Discovery 🌐

We first need to identify where the server is actually running. To this end, we will utilize several shell commands.

```bash
# Find the process and port
#
# Look for a process named "amianta-financial-services"
#
# Use 'ps' to list all processes and filter them.
# 'ps aux' shows processes owned by all users.
# '| grep -i' filters the output case-insensitively.
#
ps aux | grep -i amianta

# Find the port the server is listening on
# The server listens on a random port (1024-65535).
# 'sudo lsof -i -P -n' lists all processes with open network files.
# '| grep -i listen' filters for processes in the "LISTEN" state.
#
sudo lsof -i -P -n | grep -i listen

# You can also use netstat
# This command tells the system to list all listening TCP (Transmission Control Protocol) and UDP (User Datagram Protocol) sockets, display them with numerical addresses and port numbers, and show the process ID and program name for each one.
sudo netstat -tulnp

# Look for:
# - Processes listening on high ports (1024-65535)
# - Unusual process names

# Get process details (replace PID with actual process ID)
ps -fp [PID]

# Check its open files (might reveal configuration)
sudo lsof -p [PID]

# Try to access without authentication (will get 401 Unauthorized)
# The server is now correctly configured to handle this.
curl http://localhost:$PORT/

# To access, you must provide the correct Authorization header with the password.
# This confirms a successful connection.
curl -v -H "Authorization: <PASS>" http://localhost:$PORT/
```

That was expected, the server requires a password! But we shall crack it!

In [ ]:
def test_password(port: int, attempt: str) -> bool:
    """A pure function that tests a password attempt against the server's root path."""
    try:
        # The requests library handles the HTTP request. We use 'timeout' to prevent the script from hanging.
        response = requests.get(
            f"http://localhost:{port}/",
            headers={"Authorization": attempt},
            timeout=1
        )
        # The server returns a 200 status code only if the password is correct.
        return response.status_code == 200
    except requests.exceptions.RequestException:
        # Any connection error means the password is wrong or the server is down.
        return False

def brute_force_password(port: int, chars: str, length: int = 3) -> str:
    """Finds the correct password using a brute-force approach."""
    # Using a generator with `product` is a memory-efficient way to create all password combinations.
    attempts = ("".join(candidate) for candidate in product(chars, repeat=length))
    # `next` stops the iteration and returns the first password that passes the `test_password` function.
    return next(attempt for attempt in attempts if test_password(port, attempt))

chars = string.ascii_lowercase + string.digits
# The line below is commented out for student use, as the password should be provided.
# For the instructor, uncommenting this will demonstrate the brute-force attack.
password = brute_force_password(PORT, chars)
print(f"Password found: {password}")

Let us now explore a bit the server.

In [ ]:
# First, let's access the home directory.
# This should print the directory listing for the root ('/').s
home_response = requests.get(f"http://localhost:{PORT}/", headers={"Authorization": PASSWORD})
print("--- Home Directory Listing ---")
print(home_response.text)

In [ ]:
# Now, let's access the main transactions directory to see the folders.
# This should show a listing of 'records/' and '.secret/'.
transactions_response = requests.get(f"http://localhost:{PORT}/transactions/", headers={"Authorization": PASSWORD})
print("--- Transactions Directory Listing ---")
print(transactions_response.text)

In [ ]:
# We can check the typical filenames of records
records_response = requests.get(f"http://localhost:{PORT}/transactions/records/", headers={"Authorization": PASSWORD})
print("--- Records Directory Listing ---")
print(records_response.text)

In [ ]:
# Finally, let's look at one of the legitimate transaction files.
# This gives you a raw text example of the file structure.
# You can change the file number to see different examples.
sample_path = "/transactions/records/gov_000000.txt"
sample_response = requests.get(f"http://localhost:{PORT}{sample_path}", headers={"Authorization": PASSWORD})
print(f"--- Sample Transaction File: {sample_path} ---")
print(sample_response.text)

Hm…, a confidential tip? What might this be about, let's have a look!

In [ ]:
response = requests.get(f"http://localhost:{PORT}/confidential_tip.txt", headers={"Authorization": PASSWORD})
print(response.content.decode())

[Note]: There is "secret" directory called ```.secrets``` in ```transactions```

In [ ]:
records_in_secret_folder = requests.get(f"http://localhost:{PORT}/transactions/.secret/", headers={"Authorization": PASSWORD})
print("--- Secret Directory Listing ---")
print(records_in_secret_folder.text)

In [ ]:
secret_file_path = "/transactions/.secret/project_phoenix_0009.txt"
sample_secret_response = requests.get(f"http://localhost:{PORT}{secret_file_path}", headers={"Authorization": PASSWORD})
print(f"--- Sample Secret Transaction File: {sample_secret_response} ---")
print(sample_secret_response.text)

---

## Phase 1: Data Modeling 🧊
This cell defines our immutable `Transaction` object using Python's `dataclass` and `frozen=True`. This is a core functional programming concept as it prevents accidental modification of our data after it has been created, ensuring data integrity throughout our analysis pipeline.

In [ ]:
@dataclass(frozen=True)
class Transaction:
    path: str
    date: str
    sender: str
    receiver: str
    amount: int
    purpose: str
    reference: str
    content: str

---

## Phase 2: Building the Parsing Pipeline 📝
This is the heart of our analysis. We will create a series of small, pure functions that transform raw text from the server into our structured `Transaction` objects. Each function should have a single responsibility and not modify any global state.

1.  **`parse_line(line: str) -> Tuple[str, str]`**: A pure function that takes a line of text (e.g., `'Amount: $1,234,567'`) and returns a key-value tuple.
2.  **`create_field_updater(key: str, value: str) -> Callable`**: A higher-order function that returns a function. This inner function will take a dictionary of fields and return a *new* dictionary with the correct field updated.
3.  **`parse_transaction(port: int, password: str, path: str) -> Transaction`**: The main function that orchestrates the parsing. It will fetch a file's content and use `reduce` to apply a chain of updater functions to an initial dictionary, finally returning a `Transaction` object.

In [ ]:
def parse_line(line: str) -> Tuple[str, str]:
    """A pure function that parses a single line from a file into a key-value pair."""
    # Check if the line contains a colon to avoid errors on non-data lines.
    if ":" not in line:
        return (None, None)
    key, value = line.split(":", 1)
    return (key.strip(), value.strip())

# This function uses a "closure"; a function that is generated inside another function.
# Closures "remember" arguments passed to the outer functions, even if they are not defined in inner scope
def create_field_updater(key: str, value: str):
    """A higher-order function that returns a function to update a dictionary."""
    def update_fields(fields: dict) -> dict:
        # This function returns a new dictionary, ensuring the original is not mutated.
        updated_fields = fields.copy()
        # The `match/case` statement is a clean way to handle the different fields.
        match key:
            case "Date":
                updated_fields["date"] = value
            case "From":
                # Splits the string to get just the department name.
                updated_fields["sender"] = value.split("(")[0].strip()
            case "To":
                # Splits the string to get just the receiver name.
                updated_fields["receiver"] = value.split("(")[0].strip()
            case "Amount":
                # Sanitizes the string by removing currency symbols and commas before conversion.
                updated_fields["amount"] = int(value.replace("$", "").replace(",", ""))
            case "Purpose":
                updated_fields["purpose"] = value
            case "Reference":
                updated_fields["reference"] = value
        return updated_fields
    return update_fields

def parse_transaction(path: str, port: int, password: str) -> Transaction:
    try:
        response = requests.get(f"http://localhost:{port}{path}", headers={"Authorization": password}, timeout=5)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {path}: {e}")
        return None

    initial_fields = {
        "path": path, "date": "", "sender": "", "receiver": "",
        "amount": 0, "purpose": "", "reference": "", "content": response.text
    }
    lines = response.text.split("\n")
    updater_functions = (create_field_updater(k, v) for k, v in (parse_line(line) for line in lines) if k)
    final_fields = reduce(lambda acc, updater: updater(acc), updater_functions, initial_fields)
    return Transaction(**final_fields)

---

## Phase 3: Identifying Suspicious Activity 🕵️‍♀️
Now, let's create a predicate function that will filter our transactions. This function, `is_master_involved`, should take a `Transaction` object and return `True` if it meets any of our suspicious criteria. Think about what a 'corrupt' transaction looks like and what keywords or values would indicate its nature.

This predicate function, `is_master_involved`, uses a simple yet effective heuristic to find suspicious files. It's a pure function because it only depends on its input `transact` and has no side effects. This is a critical component of the functional pipeline.

In [ ]:
def is_master_involved(transact: Transaction) -> bool:
    search_terms = ["the master", "master's office", "the_master", "master"]
    content_lower = transact.content.lower()
    return any(term in content_lower for term in search_terms) or transact.amount > 1_000_000

---

## Phase 4: The Main Analysis Pipeline 🚀
Finally, we'll build the complete pipeline that ties everything together. This will involve using concurrent processing to handle the massive number of files efficiently and generators to conserve memory. Your main function, `analyze_all`, will:
1.  Get all file paths from the server.
2.  Use a `ThreadPoolExecutor` to concurrently parse each file using your `parse_transaction` function.
3.  Use `filter` to find all suspicious transactions.
4.  Sort the results and print the top 10 most suspicious transactions to the console.

This is the final orchestration of all our functional components. It uses generators to lazily fetch and process data, a `ThreadPoolExecutor` for concurrent requests to speed up the process, and `filter` and `sorted` to perform the final analysis. This approach is highly scalable and memory-efficient.

In [ ]:
# File path getters
def get_paths(base_path: str, port: int, password: str):
    """
    Refactored to read the total file count from the directory listing
    and generate all file paths, rather than just parsing the first few.
    """
    try:
        response = requests.get(
            f"http://localhost:{port}{base_path}",
            headers={"Authorization": password},
            timeout=5
        )
        response.raise_for_status()
        
        # Look for the 'Total files' line to get the full count.
        total_files_match = re.search(r"Total files: (\d+)", response.text)
        if total_files_match:
            total_files = int(total_files_match.group(1))

            # The naming convention for legitimate files is "gov_000000.txt"
            if "records" in base_path:
                return (
                    f"{base_path}gov_{i:06d}.txt" for i in range(total_files)
                )
            # The naming convention for corrupt files is "project_phoenix_0000.txt"
            elif "secret" in base_path:
                return (
                    f"{base_path}project_phoenix_{i:04d}.txt" for i in range(total_files)
                )

        # Fallback to an empty iterator if the count cannot be found or path is not recognized
        return iter([])

    except requests.exceptions.RequestException as e:
        print(f"Error fetching directory listing for {base_path}: {e}")
        return iter([])


# Since it takes quite some time to parse 1,000,000 files, one can also just check 
# .secret
def get_all_file_paths(port: int, password: str):
    # base_dirs = ["/transactions/records/", "/transactions/.secret/"]
    base_dirs = ["/transactions/.secret/"]
    return chain.from_iterable(get_paths(d, port, password) for d in base_dirs)

def analyze_all(port: int, password: str) -> None:
    print("Starting the corruption analysis pipeline...")
    all_paths = get_all_file_paths(port, password)
    
    # We define a simple wrapper function here. It takes a single argument (the path)
    # and internally calls `parse_transaction` with all the required context (port, password).
    parse_wrapper = partial(parse_transaction, port=port, password=password)

    # This allows us to run things in parallel, using threads
    # Threads run within the same process and are commonly used for IO-based or network tasks
    with ThreadPoolExecutor(max_workers=8) as executor:
        transactions = executor.map(parse_wrapper, all_paths)

    valid_transactions = (transact for transact in transactions if transact is not None)
    corrupt_list = list(filter(is_master_involved, valid_transactions))
    sorted_corrupt = sorted(corrupt_list, key=lambda transact: transact.amount, reverse=True)
    
    print("\nAnalysis Complete.")
    if sorted_corrupt:
        print(f"Found {len(corrupt_list)} transactions involving 'the master' or other suspicious activity.")
        for transact in sorted_corrupt:
            print("---")
            print(f"{transact.date} - {transact.sender} -> {transact.receiver}")
            print(f"   Amount: ${transact.amount:,}")
            print(f"   Purpose: {transact.purpose}")
            print(f"   File: {transact.path}")
        return sorted_corrupt
    else:
        print("No suspicious transactions found.")

---

## Analysis Results 🎉
Once your pipeline is complete, you can run the final function to see the results of your investigation.

In [ ]:
# The final call to the main analysis pipeline.
corrupt_transactions = analyze_all(port=PORT, password=PASSWORD)